In [1]:
#Create a Spark Session
from pyspark.sql import SparkSession
import findspark

findspark.init()

spark = SparkSession\
            .builder\
            .appName("SparkReadJob")\
            .config("spark.sql.shuffle.partitions", 2)\
            .config("spark.default.parallelism", 2)\
            .config("spark.sql.warehouse.dir", "spark-warehouse") \
            .enableHiveSupport() \
            .master("local[2]")\
            .getOrCreate()
print(spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/24 18:28:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


3.5.4


In [2]:
raw_data = spark\
                .read\
                .option("inferSchema", "true")\
                .option("header", "true")\
                .csv("/Users/jayandran.sampath/Documents/projects/data/data-pipelines/airflow/dags/countries_data/input/airflow_data-extract.csv")

#Print the schema for verification
raw_data.printSchema();

#Print the first 5 records for verification
raw_data.show(5)

root
 |-- Country Name: string (nullable = true)
 |-- Country Code: string (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Value: double (nullable = true)

+------------+------------+----+--------------------+
|Country Name|Country Code|Year|               Value|
+------------+------------+----+--------------------+
| Afghanistan|         AFG|2000| 3.521418059923445E9|
| Afghanistan|         AFG|2001|2.8135717538725324E9|
| Afghanistan|         AFG|2002|3.8257014389996333E9|
| Afghanistan|         AFG|2003| 4.520946818545814E9|
| Afghanistan|         AFG|2004|  5.22489671867782E9|
+------------+------------+----+--------------------+
only showing top 5 rows



In [3]:
raw_data.where(raw_data.Year >= 2010).show(5)

+------------+------------+----+--------------------+
|Country Name|Country Code|Year|               Value|
+------------+------------+----+--------------------+
| Afghanistan|         AFG|2010|1.585666855583359...|
| Afghanistan|         AFG|2011|1.780509820631408...|
| Afghanistan|         AFG|2012|1.990732977758716...|
| Afghanistan|         AFG|2013|2.014641675759867...|
| Afghanistan|         AFG|2014|2.049712855569723E10|
+------------+------------+----+--------------------+
only showing top 5 rows



In [3]:
raw_data = raw_data.drop('Country Code')
recent_data = raw_data.where(raw_data.Year >= 2015)


In [4]:
recent_data.show(5)

+------------+----+--------------------+
|Country Name|Year|               Value|
+------------+----+--------------------+
| Afghanistan|2015|1.913422164473249...|
| Afghanistan|2016|1.811657239507721...|
| Afghanistan|2017|1.875345649781586...|
| Afghanistan|2018|1.805322268741262...|
| Afghanistan|2019|1.879944449011278E10|
+------------+----+--------------------+
only showing top 5 rows



In [5]:
recent_data.createOrReplaceTempView("TempView")

In [6]:
spark.sql("""
    SELECT
        `t1`.`Country Name`,
        t1.Year,
        t1.Value As CurrentYearGDP,
        t2.Value As PreviosYearGDP,
        (t1.Value - t2.Value)/t2.Value As GDPGrowthRate
    FROM TempView t1
    LEFT JOIN TempView t2 
    ON `t1`.`Country Name` = `t2`.`Country Name`
    AND (t1.Year-1) = t2.Year
""").show(15)

+--------------------+----+--------------------+--------------------+--------------------+
|        Country Name|Year|      CurrentYearGDP|      PreviosYearGDP|       GDPGrowthRate|
+--------------------+----+--------------------+--------------------+--------------------+
|         Afghanistan|2015|1.913422164473249...|                NULL|                NULL|
|         Afghanistan|2016|1.811657239507721...|1.913422164473249...|-0.05318477378124...|
|         Afghanistan|2017|1.875345649781586...|1.811657239507721...|0.035154779218154464|
|         Afghanistan|2018|1.805322268741262...|1.875345649781586...|-0.03733891992043139|
|         Afghanistan|2019|1.879944449011278E10|1.805322268741262...|0.041334548164658196|
|         Afghanistan|2020|1.995592905214959...|1.879944449011278E10| 0.06151695400601051|
|         Afghanistan|2021|1.426649942987457...|1.995592905214959...|-0.28509971184038535|
|         Afghanistan|2022|1.450215819209039...|1.426649942987457...|0.016518331169757323|

In [ ]:
spark.sql("""
SELECT
        `t1`.`Country Name`,
        t1.Year,
        (t1.Value - t2.Value)/t2.Value AS YearOverYearChange
    FROM TempView t1
    LEFT JOIN TempView t2 ON (t1.Year-1) = t2.Year
""").show(5)

In [ ]:
recent_data.write \
      .option("compression", "gzip") \
      .partitionBy("Year") \
      .parquet(path="../dags/countries_data/stagging",
               mode="overwrite");

In [7]:
spark.stop()